# **Interpretación de modelos**

En este notebook se utilizarán las métricas de los modelos para analizar la importancia de las variables.

## **Librerias**

In [ ]:
import joblib
import pandas as pd
import plotly.express as px
from pathlib import Path

---

## **LightGBM**

In [9]:
MODELS_DIR = Path.cwd().parent / "models"

modelo_lightgbm = joblib.load(
    MODELS_DIR / "lightgbm_best_model.joblib"
)

preprocessor = modelo_lightgbm.named_steps["preprocessing"]
classifier = modelo_lightgbm.named_steps["classifier"]

features = preprocessor.get_feature_names_out()

importances = classifier.booster_.feature_importance(
    importance_type='gain'
)

dfFeatures = pd.DataFrame({
    "Variables": features,
    "Importancia": importances
})

dfFeatures_sorted = dfFeatures.sort_values(
    by="Importancia",
    ascending=False
)
print(dfFeatures_sorted)

fig = px.bar(
    dfFeatures_sorted,
    x="Variables",
    y="Importancia",
    title="Importancia de Variables - LightGBM"
)

fig.show()

                                            Variables  Importancia
8                            numericas__actualizacion  6596.544561
21  categoricas_nominales__cartera_consumo_con_lib...  6384.033021
15                           numericas__saldo_interes  4999.699053
16                           numericas__valorgarantia  4110.282408
20                            numericas__puntaje_data  3216.456582
1                              numericas__vinculacion  3182.210769
17                             numericas__ctasahorros  2765.170610
18                        numericas__curtotalingresos  1958.273905
12                             numericas__valor_cuota  1787.834698
2                                  numericas__aportes  1501.138022
14                           numericas__saldo_capital  1317.769858
22  categoricas_nominales__cartera_consumo_sin_lib...  1292.464583
13                          numericas__valor_prestamo  1264.608562
0                                    numericas__plazo  1025.89

El análisis de importancia de variables realizado con LightGBM permitió identificar cuáles características aportan mayor información al momento de predecir el default de un cliente. Para este análisis se utilizó la métrica gain, la cual mide cuánto contribuye cada variable a reducir el error del modelo durante la construcción de los árboles. A diferencia de métricas como split, que únicamente contabilizan cuántas veces una variable es utilizada, gain permite evaluar el aporte real de cada característica en el desempeño predictivo del modelo, por lo que resulta más adecuada para interpretar relevancia en problemas de clasificación.

Los resultados muestran que la variable más importante del modelo fue actualizacion, indicando que mantener la información actualizada dentro de la cooperativa tiene una fuerte relación con el comportamiento de pago de los clientes. Este hallazgo coincide con lo observado previamente en el EDA, donde los usuarios con datos desactualizados presentaban porcentajes considerablemente más altos de impago.

De igual forma, las variables relacionadas con el tipo de cartera también presentan una importancia muy alta, especialmente cartera_consumo_con_libranza, reforzando la idea de que la modalidad del crédito influye significativamente en el riesgo de default. Esto también resulta coherente con el análisis exploratorio, donde los créditos sin libranza mostraban tasas de impago mucho mayores.

Entre las variables numéricas más relevantes destacan saldo_interes, valorgarantia, puntaje_data, vinculacion y ctasahorros. En conjunto, estas variables sugieren que el modelo está capturando patrones asociados tanto al comportamiento financiero del cliente como a su estabilidad económica y relación histórica con la cooperativa. Particularmente, el puntaje_data confirma la importancia del historial crediticio como factor predictivo de incumplimiento.

Otro aspecto interesante es que variables como sexo, intestrato, grupo_edad y grupo_actividadeco presentan importancias considerablemente bajas dentro del modelo. Esto sugiere que, aunque algunas mostraban ligeras diferencias durante el EDA, su capacidad real para discriminar entre clientes con y sin default es limitada cuando se consideran conjuntamente con variables financieras más fuertes.

Finalmente, los resultados reflejan una alta coherencia entre el análisis exploratorio realizado previamente y el comportamiento final del modelo, ya que muchas de las variables que mostraban patrones diferenciadores durante el EDA terminaron siendo también las más relevantes para la predicción del default.

## **GradientBoostingClassifier**

In [10]:
MODELS_DIR = Path.cwd().parent / "models"

modelo_gbc = joblib.load(MODELS_DIR / "gbc_best_model.joblib")

preprocessor = modelo_gbc.named_steps["preprocessing"]
classifier = modelo_gbc.named_steps["classifier"]


preprocessor

classifier

# Acceder al preprocesador y al clasificador usando 'named_steps'
features = preprocessor.get_feature_names_out()
importances = classifier.feature_importances_

# Crear y ordenar el DataFrame con las importancias
dfFeatures = pd.DataFrame({"Variables": features, "Importancia": importances})
dfFeatures_sorted = dfFeatures.sort_values(by="Importancia", ascending=False)
print(dfFeatures_sorted)

fig = px.bar(
    dfFeatures_sorted,
    x="Variables",
    y="Importancia",
    title="Importancia de las Variables",
    # text="Importancia"
)

fig.show()

                                            Variables  Importancia
8                            numericas__actualizacion     0.142486
15                           numericas__saldo_interes     0.131199
22  categoricas_nominales__cartera_consumo_sin_lib...     0.094185
18                        numericas__curtotalingresos     0.090512
17                             numericas__ctasahorros     0.088813
16                           numericas__valorgarantia     0.088544
20                            numericas__puntaje_data     0.066907
12                             numericas__valor_cuota     0.055546
1                              numericas__vinculacion     0.054337
21  categoricas_nominales__cartera_consumo_con_lib...     0.040752
2                                  numericas__aportes     0.037377
14                           numericas__saldo_capital     0.029308
13                          numericas__valor_prestamo     0.022259
0                                    numericas__plazo     0.01

A diferencia de LightGBM, donde se utilizó explícitamente la métrica gain, en Gradient Boosting la importancia se calcula mediante la reducción promedio de impureza generada por cada variable a lo largo de los árboles del modelo. En términos simples, esto representa cuánto ayuda cada variable a mejorar las decisiones de clasificación dentro del conjunto de árboles.

Los resultados obtenidos muestran nuevamente una alta coherencia con el análisis exploratorio realizado previamente. La variable más importante del modelo vuelve a ser actualizacion, lo que refuerza la idea de que mantener información actualizada dentro de la cooperativa está fuertemente relacionado con un menor riesgo de impago. Este comportamiento ya había sido evidenciado durante el EDA, donde los clientes con información desactualizada presentaban tasas de default considerablemente mayores.

Asimismo, variables financieras como saldo_interes, curtotalingresos, ctasahorros, valorgarantia y puntaje_data aparecen entre las más relevantes del modelo. Esto indica que Gradient Boosting otorga un peso importante tanto a la capacidad económica del cliente como a variables relacionadas con el comportamiento financiero y crediticio.

Resulta especialmente interesante que cartera_consumo_sin_libranza tenga una importancia considerablemente alta dentro del modelo, reforzando nuevamente la hipótesis planteada en el EDA acerca de que este tipo de cartera presenta un mayor riesgo de impago frente a créditos con libranza. De esta manera, el modelo logra capturar patrones que coinciden directamente con la lógica del negocio financiero.

También se observa que variables como sexo, grupo_edad, grupo_ciudad, intestrato y garantias poseen una importancia muy baja dentro del modelo. Esto sugiere que, aunque algunas de estas variables mostraban ciertas diferencias descriptivas durante el análisis exploratorio, su capacidad predictiva real es limitada cuando el modelo dispone de variables financieras más fuertes y directamente relacionadas con el riesgo crediticio.

Finalmente, el hecho de que tanto LightGBM como Gradient Boosting coincidan en señalar variables similares como las más importantes aporta mayor confianza sobre la consistencia de los patrones encontrados en el dataset. En ambos modelos predominan variables asociadas a comportamiento financiero, historial crediticio y estabilidad económica del cliente, lo que resulta coherente con la naturaleza del problema de predicción de default.

# **Conclusiones**

En general, los resultados obtenidos muestran que tanto LightGBM como Gradient Boosting lograron identificar patrones similares dentro de los datos, lo que aporta mayor confianza sobre las variables que realmente influyen en el default de los clientes. Ambos modelos coinciden en que las variables financieras y relacionadas con el comportamiento crediticio tienen mucha más relevancia que las variables demográficas.

Una de las conclusiones más importantes es que la variable actualizacion fue la más relevante en ambos modelos, lo que indica que mantener la información actualizada de los clientes está fuertemente relacionado con un menor riesgo de impago. Además, variables como puntaje_data, saldo_interes, ctasahorros y las relacionadas con el tipo de cartera también demostraron ser fundamentales para la predicción.

Por otro lado, variables como sexo, edad o estrato tuvieron una importancia baja dentro de los modelos, lo que sugiere que estas características por sí solas no son determinantes al momento de identificar clientes con riesgo de default.

Finalmente, los resultados obtenidos mantienen una alta coherencia con el análisis exploratorio realizado previamente, ya que muchas de las variables que mostraban diferencias importantes durante el EDA terminaron siendo también las más relevantes dentro de los modelos predictivos. Esto permite concluir que el análisis realizado fue consistente y que los modelos lograron capturar adecuadamente la lógica del problema financiero estudiado.